# 9단계: 단어 인식 최종 설정 + 파이4 속도 시험 파일 만들기

08 결과: Whisper tiny 31/34(91%), base 28~29/34, small 32~33/34(97%), C 3시간 오경보는 모두 시간당 0.33회였습니다. tiny는 CPU에서 구간당 2.4초(small은 14초)입니다.
이번에는 **단어 규칙을 더 엄격하게** 고치고(예: "도와주신 거는 감사한데"가 위급으로 잡힌 오류), 다음을 확인합니다.
1. 크기별 재현율과 **오경보 목록 전체** (08에서는 small만 출력했습니다)
2. **반복 규칙**: 30초 안에 2번 이상이면 "긴급", 1번이면 "주의"로 나눴을 때의 횟수
3. **음량 게이트**: 작은 소리 구간은 Whisper에 안 보내면 처리량이 얼마나 줄고 재현율은 얼마나 떨어지는가
4. **파이4 속도 시험 파일**: 구간 24개와 시험 스크립트(`bench_asr_pi.py`)를 드라이브 `pi_bench` 폴더에 만듭니다.

**GPU(T4) 런타임**으로 10~20분 걸립니다. **4번, 5번, 6번 항목의 출력**을 복사해서 알려 주세요.

## 0. 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 파일 찾기

In [ ]:
import glob, os, re, subprocess, pickle, time, csv, unicodedata
import numpy as np
import pandas as pd

def find(name):
    hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    assert hits, f"{name}을 못 찾았어요. 드라이브에 올렸는지 확인하세요."
    return hits[0]

OWN = find("6_own")
EMB = find("embeddings_v2.npz")
OUT_DIR = os.path.dirname(EMB)
files = sorted(glob.glob(f"{OWN}/*/*.m4a") + glob.glob(f"{OWN}/*/*.wav") + glob.glob(f"{OWN}/*/*.mp3"))
print("녹음 파일", len(files), "개 / 작업 폴더:", OUT_DIR)

## 2. 캐시와 말소리 구간 준비 (고친 단어 규칙 포함)

In [ ]:
FFMPEG = "ffmpeg"
SR, HOP, WIN = 16000, 7680, 15600
SEG = HOP * 600          # 288초씩 나눠서 처리 (프레임 격자는 이어짐)

def probe_channels(path):
    r = subprocess.run([FFMPEG, "-hide_banner", "-i", path], capture_output=True, text=True, errors="replace")
    m = re.search(r"Audio: .*?, \d+ Hz, ([^,\n]+)", r.stderr)
    return 1 if (m and m.group(1).strip().startswith(("mono", "1 channel"))) else 2

def load_mono(path):
    ch = probe_channels(path)
    raw = subprocess.run([FFMPEG, "-v", "error", "-i", path, "-vn", "-ac", str(ch), "-ar", str(SR), "-f", "s16le", "-"],
                         capture_output=True).stdout
    x = np.frombuffer(raw[: len(raw) // (2 * ch) * (2 * ch)], dtype=np.int16).reshape(-1, ch)
    if ch == 1:
        return x[:, 0].copy(), "모노", None
    a, b = x[::10, 0].astype(np.float64), x[::10, 1].astype(np.float64)
    corr = float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else 1.0
    if corr < -0.3:
        return x[:, int(b.std() > a.std())].copy(), "한 채널(위상 반대)", corr
    return x.astype(np.float32).mean(axis=1).astype(np.int16), "두 채널 평균", corr

def embed_stream(x):
    embs, sp, sc, db, ts = [], [], [], [], []
    n = len(x)
    for start in range(0, n, SEG):
        seg = x[start:start + SEG + (WIN - HOP)]
        if len(seg) < WIN:
            break
        w = seg.astype(np.float32) / 32768
        scores, emb, _ = yamnet(tf.constant(w))
        scores, emb = scores.numpy(), emb.numpy()
        last = start + SEG + (WIN - HOP) >= n
        nb = len(w) // HOP
        bp = (w[: nb * HOP].reshape(nb, HOP) ** 2).mean(axis=1)
        fdb = 10 * np.log10(((bp[:-1] + bp[1:]) / 2 if nb > 1 else bp) + 1e-12)
        k = min(len(emb) if last else 600, len(emb), len(fdb))
        embs.append(emb[:k].astype(np.float16))
        sp.append(scores[:k, SPEECH].astype(np.float16))
        sc.append(scores[:k][:, SCREAM].max(axis=1).astype(np.float16))
        db.append(fdb[:k].astype(np.float32))
        ts.append(start / SR + np.arange(k) * 0.48)
    return tuple(np.concatenate(a) for a in (embs, sp, sc, db, ts))


import re, time
PKL = f"{OUT_DIR}/own_embeddings_v1.pkl"
assert os.path.exists(PKL), "own_embeddings_v1.pkl이 없어요. 07을 먼저 실행하세요."
own = pickle.load(open(PKL, "rb"))
nfc = lambda s: unicodedata.normalize("NFC", s)
for v in own.values():
    v["split"], v["name"] = nfc(v["split"]), nfc(v["name"])
E_files = [f for f, v in own.items() if "응급" in v["split"]]
C_files = [f for f, v in own.items() if ("장시간" in v["split"]) or ("연속" in v["name"]) or ("3시간" in v["name"])]
OTHER = [f for f in own if f not in E_files and f not in C_files]
assert E_files and C_files, "응급 소리/장시간 음성 녹음을 못 찾았어요."
hoursC = sum(own[f]["dur"] for f in C_files) / 3600
SPEECH_TH = 0.3

def speech_windows(v, gap):
    sp, t = v["speech"].astype(np.float32), v["t"]
    out = []
    for i in np.flatnonzero(sp > SPEECH_TH):
        if out and t[i] - t[out[-1][1]] <= gap:
            out[-1][1] = i
        else:
            out.append([i, i])
    return [(float(t[a]), float(t[b]) + 0.96) for a, b in out]

def make_windows(v, gap):
    out = []
    for k, (c0, c1) in enumerate(speech_windows(v, gap)):
        a, w1 = max(0.0, c0 - 0.5), min(v["dur"], c1 + 0.5)
        while True:
            b = min(a + 10.0, w1)
            out.append((k, a, b))
            if b >= w1 - 1e-6:
                break
            a += 9.0
    return out

WIN = {}
for f in E_files + C_files + OTHER:
    x, _, _ = load_mono(f)
    gap = 1.5 if f in E_files else 1.0
    WIN[f] = [(k, a, b, x[int(a * SR):int(b * SR)].astype(np.float32) / 32768) for k, a, b in make_windows(own[f], gap)]
    del x
n_cl = {f: len(speech_windows(own[f], 1.5)) for f in E_files}
n_E = sum(n_cl.values())
tot = sum(len(w) for w in WIN.values())
print(f"응급 발화 {n_E}개, C {hoursC:.2f}시간, 다른 녹음 {len(OTHER)}개 / 말소리 구간 합계 {tot}개 "
      f"({sum(b - a for w in WIN.values() for _, a, b, _ in w) / 60:.0f}분)")

END = r"(?:주세요|주십시오|주십쇼|줘요|줘(?!서)|주라|달라|주셔요)"
STRICT = re.compile(r"(?:^|[\s,.!?])(?:(?:도와|살려|구해)\s*%s|사람\s*살려|119\s*(?:불러|신고|에\s*전화)|일일구\s*(?:불러|신고))" % END)
LOOSE = re.compile(r"(?:^|[\s,.!?])(?:(?:[도토또]\s*[와아하]|[살쌀]려|구[해애])\s*%s|사람\s*살려|119\s*(?:불러|신고|에\s*전화)|일일구\s*(?:불러|신고))" % END)

## 3. Whisper 글자 인식 (결과는 드라이브에 저장해서 다음부터는 건너뜁니다)

In [ ]:
import torch, transformers
from transformers import pipeline
transformers.logging.set_verbosity_error()
dev = 0 if torch.cuda.is_available() else -1
print("런타임:", "GPU" if dev == 0 else "CPU (오래 걸려요)")
GEN = {"language": "ko", "task": "transcribe", "max_new_tokens": 48}
SIZES = ("tiny", "base", "small")
TXT = {}

def run(asr, wins):
    out = []
    for s in range(0, len(wins), 128):
        batch = [{"raw": arr, "sampling_rate": SR} for _, _, _, arr in wins[s:s + 128]]
        out += [r["text"] for r in asr(batch, batch_size=16, generate_kwargs=GEN)]
    return out

for short in SIZES:
    pk = f"{OUT_DIR}/asr_texts_{short}.pkl"
    if os.path.exists(pk):
        TXT[short] = pickle.load(open(pk, "rb")); print(short, "캐시 사용"); continue
    asr = pipeline("automatic-speech-recognition", model=f"openai/whisper-{short}", device=dev,
                   torch_dtype=torch.float16 if dev == 0 else torch.float32)
    TXT[short] = {f: run(asr, WIN[f]) for f in E_files + C_files + OTHER}
    pickle.dump(TXT[short], open(pk, "wb")); print(short, "완료", flush=True)
    del asr
    if dev == 0:
        torch.cuda.empty_cache()

## 4. 크기별 재현율·오경보와 목록
- 규칙: **엄격**은 "도와주세요/도와줘/살려줘/구해줘/사람 살려/119 불러" 같은 완성된 표현만, **느슨**은 "토아주세요" 같은 철자 변형도 인정합니다. ("도와주신", "도와줘서"는 인정하지 않습니다.)
- 아래 목록에서 **정상 말인데 위급으로 잡힌 것**과 **놓친 응급 발화의 글자**를 확인하세요.

In [ ]:
pd.set_option("display.width", 200)
mmss = lambda s: f"{int(s) // 3600}:{int(s) % 3600 // 60:02d}:{int(s) % 60:02d}"

def hits_E(texts, rx):
    return sum(any(rx.search(tx) for (kk, _, _, _), tx in zip(WIN[f], texts[f]) if kk == k) for f in E_files for k in range(n_cl[f]))

rows = []
for short in SIZES:
    for nm, rx in (("엄격", STRICT), ("느슨", LOOSE)):
        c = sum(bool(rx.search(tx)) for f in C_files for tx in TXT[short][f])
        o = sum(bool(rx.search(tx)) for f in OTHER for tx in TXT[short][f])
        by_dev = [sum(any(rx.search(tx) for (kk, _, _, _), tx in zip(WIN[f], TXT[short][f]) if kk == k) for k in range(n_cl[f])) for f in E_files]
        rows.append({"크기": short, "규칙": nm, "E재현율": f"{hits_E(TXT[short], rx)}/{n_E}", "파일별E": "/".join(map(str, by_dev)),
                     "C횟수": c, "C시간당": round(c / hoursC, 2), "다른녹음횟수": o})
res = pd.DataFrame(rows)
print(res.to_string(index=False))
print("(파일별E 순서:", [own[f]["name"][:14] for f in E_files], ")")

print("\n[정상 말인데 위급으로 인식된 구간 (엄격 규칙)]")
for short in SIZES:
    for f in C_files + OTHER:
        for (k, a, b, _), tx in zip(WIN[f], TXT[short][f]):
            if STRICT.search(tx):
                print(f"  {short} {own[f]['split']}/{own[f]['name'][:18]} {mmss(a)}  {tx.strip()[:48]}")
print("\n[놓친 응급 발화: 인식된 글자 (느슨 규칙)]")
for short in SIZES:
    for f in E_files:
        for k in range(n_cl[f]):
            ws = [(a, tx) for (kk, a, _, _), tx in zip(WIN[f], TXT[short][f]) if kk == k]
            if not any(LOOSE.search(tx) for _, tx in ws):
                print(f"  {short} {own[f]['name'][:14]} {mmss(ws[0][0])[2:]}  {' '.join(tx.strip() for _, tx in ws)[:30]}")

## 5. 반복 규칙 (엄격 규칙 기준)
위급 단어가 인식된 시각을 모아서, **직전 인식에서 30초 안에 또 인식되면 "긴급", 아니면 "주의"** 로 셉니다. (같은 발화가 구간 둘로 나뉜 중복은 3초 안이면 하나로 봅니다.)
응급 녹음은 5초 간격으로 계속 외친 것이니 "긴급"이 많이 나와야 하고, C와 다른 녹음에서는 "긴급"이 0이고 "주의"가 적어야 합니다.

In [ ]:
def classify(times):
    kept = []
    for x in sorted(times):
        if not kept or x - kept[-1] >= 3.0:
            kept.append(x)
    att = urg = 0
    for i, x in enumerate(kept):
        if i and x - kept[i - 1] <= 30.0:
            urg += 1
        else:
            att += 1
    return att, urg

def hit_times(short, f):
    return [a for (_, a, _, _), tx in zip(WIN[f], TXT[short][f]) if STRICT.search(tx)]

rows = []
for short in SIZES:
    r = {"크기": short}
    for nm, fl in (("응급", E_files), ("C", C_files), ("다른녹음", OTHER)):
        a = u = 0
        for f in fl:
            x, y = classify(hit_times(short, f)); a += x; u += y
        r[nm + " 주의"], r[nm + " 긴급"] = a, u
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

## 6. 음량 게이트
말소리 구간의 **가장 큰 음량**(dBFS)이 기준보다 작으면 Whisper에 보내지 않는다고 가정합니다. 위급한 외침은 크게 나는 경향이 있으니, 작은 구간을 걸러서 파이의 부담을 줄일 수 있는지 봅니다.
`C구간/시간`은 Whisper가 처리해야 하는 구간 수(한 시간당)입니다.

In [ ]:
def wdb(f, a, b):
    v = own[f]; m = (v["t"] >= a - 0.96) & (v["t"] <= b)
    return float(v["db"][m].max()) if m.any() else -120.0

DB = {f: np.array([wdb(f, a, b) for _, a, b, _ in WIN[f]]) for f in E_files + C_files + OTHER}
pe = np.concatenate([DB[f] for f in E_files]); pc = np.concatenate([DB[f] for f in C_files])
print(f"응급 발화 구간 음량(dBFS): 최소 {pe.min():.0f} / 하위10% {np.percentile(pe, 10):.0f} / 중앙 {np.median(pe):.0f}")
print(f"C 말소리 구간 음량(dBFS): 중앙 {np.median(pc):.0f} / 상위10% {np.percentile(pc, 90):.0f}\n")

rows = []
for short in ("tiny", "small"):
    for g in (-120, -40, -35, -30, -25, -20):
        e = sum(any(STRICT.search(tx) and DB[f][i] >= g for i, ((kk, _, _, _), tx) in enumerate(zip(WIN[f], TXT[short][f])) if kk == k)
                for f in E_files for k in range(n_cl[f]))
        c = sum(bool(STRICT.search(tx)) and DB[f][i] >= g for f in C_files for i, tx in enumerate(TXT[short][f]))
        o = sum(bool(STRICT.search(tx)) and DB[f][i] >= g for f in OTHER for i, tx in enumerate(TXT[short][f]))
        rows.append({"크기": short, "게이트": "없음" if g == -120 else g, "C구간/시간": round(sum((DB[f] >= g).sum() for f in C_files) / hoursC),
                     "E재현율": f"{e}/{n_E}", "C횟수": c, "다른녹음횟수": o})
print(pd.DataFrame(rows).to_string(index=False))

## 7. 파이4 속도 시험 파일 만들기
응급 발화 12개와 C 일반 구간 12개를 16kHz WAV로 잘라 드라이브의 `pi_bench` 폴더에 저장하고, 시험 스크립트 `bench_asr_pi.py`도 같이 넣습니다.
이 폴더를 **파이4로 복사**해서 `pip install faster-whisper numpy` 후 `python3 bench_asr_pi.py pi_bench`를 실행하면 됩니다.

In [ ]:
PI_SCRIPT = '"""파이에서 Whisper(tiny/base/small) 처리 속도 시험: 마이크 없이 WAV 구간 파일로 잽니다.\n\n준비:  pip install faster-whisper numpy   (처음 실행할 때 모델을 내려받으므로 인터넷이 필요합니다)\n사용법: python3 bench_asr_pi.py pi_bench [--threads 4] [--sizes tiny,base,small]\n        (pi_bench 폴더는 07·09 노트북이 드라이브에 만든 것을 파이로 복사한 것)\n출력:   구간당 처리 시간, 실시간 대비 배율(1보다 작으면 실시간보다 빠름), 온도·성능 제한 표시, 인식된 글자\n"""\nimport argparse\nimport csv\nimport re\nimport subprocess\nimport time\nimport wave\nfrom pathlib import Path\n\nimport numpy as np\nfrom faster_whisper import WhisperModel\n\nEND = r"(?:주세요|주십시오|주십쇼|줘요|줘(?!서)|주라|달라|주셔요)"\nKEY = re.compile(r"(?:^|[\\s,.!?])(?:(?:도와|살려|구해)\\s*%s|사람\\s*살려|119\\s*(?:불러|신고|에\\s*전화)|일일구\\s*(?:불러|신고))" % END)\n\nap = argparse.ArgumentParser()\nap.add_argument("folder")\nap.add_argument("--threads", type=int, default=4)\nap.add_argument("--sizes", default="tiny,base,small")\nargs = ap.parse_args()\n\n\ndef sh(cmd):\n    try:\n        return subprocess.run(cmd, capture_output=True, text=True).stdout.strip() or "?"\n    except Exception:\n        return "?"\n\n\ndef read_wav(p):\n    with wave.open(str(p), "rb") as w:\n        assert w.getframerate() == 16000 and w.getnchannels() == 1 and w.getsampwidth() == 2, p\n        return np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16).astype(np.float32) / 32768\n\n\nfolder = Path(args.folder)\nitems = list(csv.DictReader(open(folder / "bench_list.csv", encoding="utf-8-sig")))\naudio = [read_wav(folder / r["file"]) for r in items]\nprint(f"구간 {len(items)}개 (합계 {sum(len(a) for a in audio) / 16000:.0f}초), 스레드 {args.threads}")\nprint("시작 온도:", sh(["vcgencmd", "measure_temp"]), "| 성능제한:", sh(["vcgencmd", "get_throttled"]))\n\nfor size in args.sizes.split(","):\n    model = WhisperModel(size, device="cpu", compute_type="int8", cpu_threads=args.threads)\n    list(model.transcribe(audio[0], language="ko", beam_size=1)[0])          # 워밍업(시간에 포함 안 함)\n    times, e_hit, c_hit, lines = [], 0, 0, []\n    for r, a in zip(items, audio):\n        t0 = time.time()\n        segs, _ = model.transcribe(a, language="ko", beam_size=1, vad_filter=False, condition_on_previous_text=False)\n        text = " ".join(s.text.strip() for s in segs)\n        times.append(time.time() - t0)\n        hit = bool(KEY.search(text))\n        if r["source"] == "E":\n            e_hit += hit\n        else:\n            c_hit += hit\n        lines.append(f"  [{r[\'source\']}] {len(a) / 16000:4.1f}초 -> {times[-1]:5.2f}초 | {\'단어\' if hit else \'    \'} | {text[:34]}")\n    n_e = sum(r["source"] == "E" for r in items)\n    total_audio = sum(len(a) for a in audio) / 16000\n    print(f"\\n=== {size} (int8) ===")\n    print("\\n".join(lines))\n    print(f"평균 {np.mean(times):.2f}초/구간, 최대 {max(times):.2f}초, 실시간 대비 {sum(times) / total_audio:.2f}배 "\n          f"| 위급 구간 인식 {e_hit}/{n_e}, 일반 구간 오인식 {c_hit}/{len(items) - n_e}")\n    print("온도:", sh(["vcgencmd", "measure_temp"]), "| 성능제한:", sh(["vcgencmd", "get_throttled"]), "(0x0이면 제한 없음)")\n'
import wave
bench = f"{OUT_DIR}/pi_bench"
os.makedirs(bench, exist_ok=True)

def pick(items, n):
    idx = np.linspace(0, len(items) - 1, min(n, len(items))).astype(int)
    return [items[i] for i in sorted(set(idx))]

E_w = [(f, w) for f in E_files for w in WIN[f]]
C_w = [(f, w) for f in C_files for w in WIN[f] if 3.0 <= w[2] - w[1] <= 10.0]
rows = []
for src, items in (("E", pick(E_w, 12)), ("C", pick(C_w, 12))):
    for j, (f, (k, a, b, arr)) in enumerate(items):
        name = f"{src}_{j:02d}.wav"
        with wave.open(f"{bench}/{name}", "wb") as w:
            w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR)
            w.writeframes(np.clip(arr * 32768, -32768, 32767).astype(np.int16).tobytes())
        rows.append({"file": name, "source": src, "seconds": round(b - a, 2)})
pd.DataFrame(rows).to_csv(f"{bench}/bench_list.csv", index=False, encoding="utf-8-sig")
open(f"{bench}/bench_asr_pi.py", "w", encoding="utf-8").write(PI_SCRIPT)
print(f"저장: {bench}  (구간 {len(rows)}개, 합계 {sum(r['seconds'] for r in rows):.0f}초)")

## 8. 결과 저장

In [ ]:
res.to_csv(f"{OUT_DIR}/result_asr_final.csv", index=False, encoding="utf-8-sig")
print("저장 완료")